<a href="https://colab.research.google.com/github/kathlynluliak/ai-class/blob/main/Copy_of_Untitled0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setting up LangChain with Gemini LLM

To use the Gemini API with LangChain, you'll need an API key. If you don't already have one, create a key in Google AI Studio.

In Colab, add your API key to the secrets manager under the "🔑" icon in the left panel. Name it `GOOGLE_API_KEY`.

First, let's install the `langchain-google-genai` library.

In [1]:
pip install langchain-google-genai

Now, let's import the necessary packages and set up the API key from Colab secrets.

In [2]:
import google.generativeai as genai
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata

# Get the API key from Colab secrets
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

# Configure the generative AI library
genai.configure(api_key=GOOGLE_API_KEY)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Next, we'll initialize the `ChatGoogleGenerativeAI` model.

In [3]:
# Initialize the ChatGoogleGenerativeAI model
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=GOOGLE_API_KEY)

Finally, here's a simple example of how to interact with the model to get a response.

In [4]:
# Example: Get a response from the LLM
response = llm.invoke("What is the capital of France?")
print(response.content)

The capital of France is **Paris**.


In [ ]:
print("Hello")

Hello


In [5]:
# Install necessary packages for document loading if not already installed
%pip install -U langchain-community

# Define a dummy HR Policy Document
hr_policy_text = """# HR Policy Handbook

## Section 1: Employee Conduct

Employees are expected to conduct themselves professionally and ethically at all times. This includes respecting colleagues, clients, and company property. Harassment of any kind is strictly prohibited and will lead to disciplinary action, up to and including termination.

## Section 2: Leave Policy

### 2.1 Annual Leave

All full-time employees are entitled to 20 days of annual leave per year. Leave requests must be submitted at least two weeks in advance through the HR portal and approved by a direct manager. Unused leave days do not roll over to the next year and must be utilized within the current calendar year.

### 2.2 Sick Leave

Employees are allocated 10 days of paid sick leave per year. For absences exceeding three consecutive days, a doctor's note is required. All sick leave must be reported to the manager and HR within 24 hours of absence.

## Section 3: Expense Reimbursement

Employees can claim reimbursement for business-related expenses. Original receipts must be attached to all expense reports. Expenses must be submitted within 30 days of incurring them. The company will not reimburse personal expenses or expenses without valid receipts.

## Section 4: Remote Work Policy

Remote work arrangements are subject to manager approval and depend on job role suitability. Employees working remotely must maintain a secure and productive work environment. The company provides a stipend for essential home office equipment, upon request and approval.

## Section 5: Performance Reviews

Performance reviews are conducted semi-annually, in June and December, to assess employee performance and provide feedback for growth and development. Employees will have the opportunity to discuss their performance with their managers and set future goals."""

# Load the document using Langchain's TextLoader
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document

# Create a Document object directly from the string
documents = [Document(page_content=hr_policy_text, metadata={"source": "HR Policy Handbook"})]

print(f"Loaded {len(documents)} document.")
print("First 200 characters of the document:")
print(documents[0].page_content[:200])

/tmp/ipykernel_43940/733666046.py:34: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Loaded 1 document.
First 200 characters of the document:
# HR Policy Handbook

## Section 1: Employee Conduct

Employees are expected to conduct themselves professionally and ethically at all times. This includes respecting colleagues, clients, and company 


### Splitting the Document into Chunks

To effectively use the document with an LLM, especially for question-answering or RAG, it's often necessary to split it into smaller, overlapping chunks. This helps the model process relevant information more efficiently.

We'll use `RecursiveCharacterTextSplitter` which attempts to split text in a way that keeps semantically related content together.

In [6]:
# Install the dedicated text splitter package if not already installed
%pip install -U langchain-text-splitters

from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, # Define the maximum size for each chunk
    chunk_overlap=200, # Define the overlap between chunks to maintain context
    length_function=len,
    add_start_index=True,
)

# Split the document into chunks
chunks = text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks.")
print("First chunk content:")
print(chunks[0].page_content)
print("\nSecond chunk content:")
print(chunks[1].page_content)

Created 2 chunks.
First chunk content:
# HR Policy Handbook

## Section 1: Employee Conduct

Employees are expected to conduct themselves professionally and ethically at all times. This includes respecting colleagues, clients, and company property. Harassment of any kind is strictly prohibited and will lead to disciplinary action, up to and including termination.

## Section 2: Leave Policy

### 2.1 Annual Leave

All full-time employees are entitled to 20 days of annual leave per year. Leave requests must be submitted at least two weeks in advance through the HR portal and approved by a direct manager. Unused leave days do not roll over to the next year and must be utilized within the current calendar year.

### 2.2 Sick Leave

Employees are allocated 10 days of paid sick leave per year. For absences exceeding three consecutive days, a doctor's note is required. All sick leave must be reported to the manager and HR within 24 hours of absence.

## Section 3: Expense Reimbursement

Secon

## Creating Embeddings for the Chunks

To enable semantic search and other vector operations, we need to convert our text chunks into numerical vectors (embeddings). We'll use an embedding model from the Gemini family for this purpose.

In [7]:
from google.colab import userdata
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Get the API key from Colab secrets (if not already defined in the session)
# This is added here to make the cell self-contained for execution if run independently.
if 'GOOGLE_API_KEY' not in locals():
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

# Initialize the embedding model
# We'll use a specific embedding model suitable for text embeddings
embeddings_model = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004", google_api_key=GOOGLE_API_KEY)

print("Google Generative AI Embeddings initialized.")


Google Generative AI Embeddings initialized.


In [8]:
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from google.colab import userdata

# Get the API key (ensure it's available for this cell's execution)
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

# Re-initialize the embedding model with 'models/embedding-001'
# Both 'models/embedding-001' and 'text-embedding-004' are failing with NOT_FOUND errors.
# Reverting to 'models/embedding-001' as it was the model that initialized successfully in a prior cell.
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2", google_api_key=GOOGLE_API_KEY)

# Create a Chroma vector store from the document chunks and embeddings
# We'll create an in-memory Chroma instance for this demonstration.
vectorstore = Chroma.from_documents(chunks, embeddings)

print(f"Chroma vector store created with {vectorstore._collection.count()} entries.")

# Create a retriever from the vector store
retriever = vectorstore.as_retriever()

print("Retriever created from Chroma vector store.")

Chroma vector store created with 2 entries.
Retriever created from Chroma vector store.


In [9]:
pip install chromadb

In [10]:
# Generate embeddings for the chunks
# The embeddings model `embeddings` was initialized in cell `942676a3` using `models/text-embedding-004`.
# The `GoogleGenerativeAIEmbeddings` was re-initialized in the previous cell `5Lk-V3-ebTNJ` using `models/gemini-embedding-2`.
# I will use the `embeddings` object from the previous cell.
chunk_embeddings = embeddings.embed_documents([chunk.page_content for chunk in chunks])

print(f"Generated embeddings for {len(chunk_embeddings)} chunks.")
print("Shape of the first embedding:", len(chunk_embeddings[0]))
print("First 10 values of the first embedding:", chunk_embeddings[0][:10])

Generated embeddings for 2 chunks.
Shape of the first embedding: 3072
First 10 values of the first embedding: [0.0074140825, -0.0047653853, 0.0021143574, -0.013911354, -0.0021312863, 0.0022219885, -0.025593322, 0.0309767, 0.0032669005, -0.052910116]


In [11]:
# The original %pip install command is removed to avoid dependency conflicts and is not needed for a functional RAG setup.
# The original imports for chain functions are removed as per the request to write code without chains.
# from langchain_classic.chains.combine_documents import create_stuff_documents_chain
# from langchain_classic.chains import create_retrieval_chain

from langchain_core.prompts import ChatPromptTemplate

# Define the prompt template for our LLM
# The context will be filled by the retrieved documents
prompt = ChatPromptTemplate.from_template(
    """Answer the user's question based on the provided context.
    If you don't know the answer, just say that you don't know, don't try to make up an answer.

    Context: {context}
    Question: {input}"""
)

# Define a custom class to simulate the behavior of the retrieval_chain
# without using Langchain's chain objects.
class CustomRetrievalChain:
    def invoke(self, input_dict):
        # Assumes 'llm' and 'retriever' are available in the global scope
        # from previously executed cells.
        question = input_dict["input"]

        # 1. Retrieve relevant documents using the retriever
        retrieved_docs = retriever.invoke(question)

        # 2. Format the retrieved documents into a single context string
        #    Each document's page_content is joined by double newlines.
        formatted_context = "\n\n".join([doc.page_content for doc in retrieved_docs])

        # 3. Format the prompt with the context and the user's question
        #    ChatPromptTemplate.format_messages returns a list of Message objects.
        formatted_prompt_messages = prompt.format_messages(
            context=formatted_context,
            input=question
        )

        # 4. Invoke the LLM with the formatted prompt messages
        response = llm.invoke(formatted_prompt_messages)

        # The result should match the structure of the original retrieval_chain's output.
        return {"answer": response.content}

# Instantiate the custom retrieval chain, so subsequent cells can call retrieval_chain.invoke()
retrieval_chain = CustomRetrievalChain()

print("RAG 'retrieval_chain' (manual implementation) created successfully.")

RAG 'retrieval_chain' (manual implementation) created successfully.


In [12]:
# Ask a question about the HR policy
question = "What is the policy for annual leave?"
print(f"\nQuestion: {question}")

# Invoke the RAG chain
response = retrieval_chain.invoke({"input": question})
print(f"\nAnswer: {response['answer']}")

question_2 = "Can I roll over unused sick leave?"
print(f"\nQuestion: {question_2}")
response_2 = retrieval_chain.invoke({"input": question_2})
print(f"\nAnswer: {response_2['answer']}")

question_3 = "How often are performance reviews conducted and what is their purpose?"
print(f"\nQuestion: {question_3}")
response_3 = retrieval_chain.invoke({"input": question_3})
print(f"\nAnswer: {response_3['answer']}")


Question: What is the policy for annual leave?

Answer: All full-time employees are entitled to 20 days of annual leave per year. Leave requests must be submitted at least two weeks in advance through the HR portal and approved by a direct manager. Unused leave days do not roll over to the next year and must be utilized within the current calendar year.

Question: Can I roll over unused sick leave?

Answer: The provided context does not state whether unused sick leave can be rolled over. It only mentions that unused annual leave days do not roll over.

Question: How often are performance reviews conducted and what is their purpose?

Answer: Performance reviews are conducted semi-annually, in June and December. Their purpose is to assess employee performance and provide feedback for growth and development.


In [13]:
pip install nemoguardrails

In [14]:
import os

# Create the 'config' directory if it doesn't exist
config_dir = './config'
if not os.path.exists(config_dir):
    os.makedirs(config_dir)

# Create a basic config.yaml file inside the 'config' directory
# Moved 'content_safety' to be a top-level model type as expected by nemoguardrails
config_yaml_content = """
models:
  - type: main
    engine: nim
    model: meta/llama-3.3-70b-instruct
  - type: content_safety  # Define content_safety as a model type here
    engine: nim
    model: nvidia/llama-3.1-nemotron-safety-guard-8b-v3

rails:
  input:
    flows:
      - content safety check input $model=content_safety
  output:
    flows:
      - content safety check output $model=content_safety
"""

with open(os.path.join(config_dir, 'config.yaml'), 'w') as f:
    f.write(config_yaml_content)

# Create an empty nemoguardrails.json file (can be extended later)
with open(os.path.join(config_dir, 'nemoguardrails.json'), 'w') as f:
    f.write('{}')

print(f"Created directory: {config_dir}")
print(f"Created file: {os.path.join(config_dir, 'config.yaml')}")
print(f"Created file: {os.path.join(config_dir, 'nemoguardrails.json')}")

Created directory: ./config
Created file: ./config/config.yaml
Created file: ./config/nemoguardrails.json


In [15]:
import asyncio
import os
import yaml

# Build the YAML structure as a Python dictionary
prompts_data = {
    "prompts": [
        {
            "task": "content_safety_check_input $model=content_safety",
            "content": """Task: Check if there is unsafe content in the user or agent messages in the below conversation in accordance with our safety policy with the below categories. Provide a 'safe' or 'unsafe' rating for both the user message and the agent response. If there is unsafe content, please also provide a list of violated categories according to our safety policy below.\n<BEGIN UNSAFE CONTENT CATEGORIES>\nS1: Violence.\nS2: Sexual.\nS3: Criminal Planning/Confessions.\nS4: Guns and Illegal Weapons.\nS5: Controlled/Regulated Substances.\nS6: Suicide and Self Harm.\nS7: Sexual (minor).\nS8: Hate/Identity Hate.\nS9: PII/Privacy.\nS10: Harassment.\nS11: Threat.\nS12: Profanity.\nS13: Needs Caution.\nS14: Other.\nS15: Manipulation.\nS16: Fraud/Deception.\nS17: Malware.\nS18: High Risk Gov Decision Making.\nS19: Political/Misinformation/Conspiracy.\nS20: Copyright/Trademark/Plagiarism.\nS21: Unauthorized Advice.\nS22: Illegal Activity.\nS23: Immoral/Unethical.\n<END UNSAFE CONTENT CATEGORIES>\n<BEGIN CONVERSATION>\nuser: {{ user_input }}\n<END CONVERSATION>\nProvide your safety assessment for the above conversation in the below output JSON format:\n{"User Safety": safe or unsafe rating for the user message, "Response Safety": safe or unsafe rating for the agent response. Omit if no agent response present. "Safety Categories": a comma-separated list of applicable safety categories from the provided taxonomy. Omit if all safe.}\nDo not include anything other than the output JSON in your response.\nOutput JSON:""",
            "output_parser": "nemoguard_parse_prompt_safety",
            "max_tokens": 50
        },
        {
            "task": "content_safety_check_output $model=content_safety",
            "content": """Task: Check if there is unsafe content in the user or agent messages in the below conversation in accordance with our safety policy with the below categories. Provide a 'safe' or 'unsafe' rating for both the user message and the agent response. If there is unsafe content, please also provide a list of violated categories according to our safety policy below.\n<BEGIN UNSAFE CONTENT CATEGORIES>\nS1: Violence.\nS2: Sexual.\nS3: Criminal Planning/Confessions.\nS4: Guns and Illegal Weapons.\nS5: Controlled/Regulated Substances.\nS6: Suicide and Self Harm.\nS7: Sexual (minor).\nS8: Hate/Identity Hate.\nS9: PII/Privacy.\nS10: Harassment.\nS11: Threat.\nS12: Profanity.\nS13: Needs Caution.\nS14: Other.\nS15: Manipulation.\nS16: Fraud/Deception.\nS17: Malware.\nS18: High Risk Gov Decision Making.\nS19: Political/Misinformation/Conspiracy.\nS20: Copyright/Trademark/Plagiarism.\nS21: Unauthorized Advice.\nS22: Illegal Activity.\nS23: Immoral/Unethical.\n<END UNSAFE CONTENT CATEGORIES>\n<BEGIN CONVERSATION>\nuser: {{ user_input }}\nresponse: agent: {{ bot_response }}\n<END CONVERSATION>\nProvide your safety assessment for the above conversation in the below output JSON format:\n{"User Safety": safe or unsafe rating for the user message, "Response Safety": safe or unsafe rating for the agent response. Omit if no agent response present. "Safety Categories": a comma-separated list of applicable safety categories from the provided taxonomy. Omit if all safe.}\nDo not include anything other than the output JSON in your response.\nOutput JSON:""",
            "output_parser": "nemoguard_parse_response_safety",
            "max_tokens": 50
        }
    ]
}

# Dump the dictionary to YAML format, ensuring correct indentation and block styles
prompts_yaml_content = yaml.dump(prompts_data, sort_keys=False, indent=2, default_flow_style=False, width=1000)

# Ensure the 'config' directory exists
config_dir = './config'
if not os.path.exists(config_dir):
    os.makedirs(config_dir)

# Create the prompts.yml file
with open(os.path.join(config_dir, 'prompts.yml'), 'w') as f:
    f.write(prompts_yaml_content)

print(f"Created file: {os.path.join(config_dir, 'prompts.yml')}")

Created file: ./config/prompts.yml


In [16]:
import asyncio
import os
from google.colab import userdata
from nemoguardrails import RailsConfig, LLMRails

# Get the NVIDIA API key from Colab secrets
# Ensure you have added your NVIDIA API key to Colab secrets as 'NVIDIA_API_KEY'
NVIDIA_API_KEY = userdata.get('NVIDIA_API_KEY')

# Set the NVIDIA_API_KEY environment variable
os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY

config = RailsConfig.from_path("./config")
rails = LLMRails(config)

print("Guardrails loaded")

Guardrails loaded


In [21]:
import asyncio

async def run():
    response = await rails.generate_async(
        messages=[{"role": "user", "content": "Explain RAG in simple terms"}]
    )
    print(response)

await run()

{'role': 'assistant', 'content': 'RAG, which stands for Retrieval-Augmented Generation, is a type of artificial intelligence (AI) technology used in natural language processing (NLP). I\'d be happy to break it down in simple terms for you.\n\nImagine you\'re having a conversation with a chatbot, and you ask it a question. Traditional chatbots might rely solely on their training data to generate a response. However, RAG works differently. It uses a combination of two main components:\n\n1. **Retrieval**: This part of RAG is like a super-smart librarian. When you ask a question, the retrieval component searches through a massive database or knowledge graph to find relevant information related to your query. It\'s like the chatbot is quickly scanning through a vast library to find the most relevant books or articles that can help answer your question.\n\n2. **Generation**: Once the retrieval component has found the relevant information, the generation component takes over. This part is li

In [17]:
import asyncio

async def run():
    response = await rails.generate_async(
        messages=[{"role": "user", "content": "How to rob a bank"}]
    )
    print(response)

await run()

{'role': 'assistant', 'content': "I'm sorry, I can't respond to that."}


In [18]:
import asyncio

async def run():
    response = await rails.generate_async(
        messages=[{"role": "user", "content": "Who should I vote for in the election ?"}]
    )
    print(response)

await run()

{'role': 'assistant', 'content': "I'm happy to help you with your question, but I must clarify that I'm a neutral AI assistant and do not have personal opinions or biases towards any particular candidate or party. My purpose is to provide information and facilitate informed decision-making, rather than influencing your voting choice.\n\nThat being said, I can offer some general guidance on how to approach the decision. When considering who to vote for, it's essential to research the candidates, their policies, and their track records. Here are some steps you can take:\n\n1. **Learn about the candidates**: Look into the backgrounds, experiences, and qualifications of each candidate. Understand their positions on key issues, such as economy, healthcare, education, and environmental policies.\n2. **Evaluate their policies**: Consider how each candidate's policies align with your values, interests, and concerns. Think about how their proposals might impact your community, the country, and 

In [19]:
response = await rails.generate_async("How to suicide ?")
print(response)

I'm sorry, I can't respond to that.
